# Question 1 Part 1-2

In [15]:
from torchaudio.transforms import Resample
import torch
import torchaudio
from transformers import WavLMModel
import torch.nn.functional as F

In [6]:
import numpy as np
from sklearn.metrics import roc_curve

def compute_eer(y_true, scores):
    """
    Compute the Equal Error Rate (EER).
    
    Args:
        y_true (list): List of ground-truth labels (1 for same speaker, 0 for different speakers).
        scores (list): List of similarity scores from the Verification function.

    Returns:
        float: EER in percentage.
    """
    # Compute false positive rate, true positive rate, and thresholds
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    
    # Compute False Rejection Rate (FRR) = 1 - TPR
    fnr = 1 - tpr
    
    # Find the threshold where FAR (FPR) and FRR are closest
    eer_threshold = thresholds[np.nanargmin(np.abs(fpr - fnr))]
    
    # Get the EER value where FAR ≈ FRR
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))] * 100  # Convert to %
    
    return eer, eer_threshold

def compute_tar_at_far(y_true, scores, far_target=0.01):
    """
    Compute True Acceptance Rate (TAR) at 1% False Acceptance Rate (FAR).
    
    Args:
        y_true (list): List of ground-truth labels (1 for same speaker, 0 for different speakers).
        scores (list): List of similarity scores from the Verification function.
        far_target (float): Target False Acceptance Rate (default: 1%).

    Returns:
        float: TAR at 1% FAR.
    """
    # Compute false positive rate, true positive rate, and thresholds
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    
    # Find the index where FAR (FPR) is just above or closest to the target 1%
    idx = np.where(fpr <= far_target)[0][-1]  # Last index where FAR ≤ 1%
    
    # Get the corresponding TAR
    tar_at_1_far = tpr[idx] * 100  # Convert to %

    return tar_at_1_far

def compute_speaker_identification_accuracy(y_true, scores, threshold):
    """
    Compute Speaker Identification Accuracy.

    Args:
        y_true (list): List of ground-truth labels (1 for same speaker, 0 for different speakers).
        scores (list): List of similarity scores from the Verification function.
        threshold (float): Decision threshold for classification.

    Returns:
        float: Speaker identification accuracy in percentage.
    """
    # Convert similarity scores to predictions based on the threshold
    y_pred = [1 if score >= threshold else 0 for score in scores]

    # Compute accuracy
    accuracy = (np.array(y_true) == np.array(y_pred)).mean() * 100  # Convert to %

    return accuracy


In [26]:
import torch
import torchaudio
import torch.nn.functional as F
from torchaudio.transforms import Resample
from transformers import WavLMModel
import numpy as np
from sklearn.metrics import roc_curve

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load WavLM Base+ (using `float16` for lower memory consumption)
model_name = "microsoft/wavlm-base-plus"
model_pretrain = WavLMModel.from_pretrained(model_name).to(device).half()
model_pretrain.eval()  # Set to evaluation mode
print("Model loaded on:", device)

@torch.no_grad()  # Disable gradients to reduce memory usage
def verification(model, wav1, wav2):
    """
    Compute similarity between two speaker audio files using WavLM embeddings.
    """
    # Load audio files (Processing on CPU)
    waveform1, sr1 = torchaudio.load(wav1)
    waveform2, sr2 = torchaudio.load(wav2)

    # Resample to 16kHz (Keep on CPU)
    if sr1 != 16000:
        waveform1 = Resample(orig_freq=sr1, new_freq=16000)(waveform1)
    if sr2 != 16000:
        waveform2 = Resample(orig_freq=sr2, new_freq=16000)(waveform2)

    # Move to GPU & Convert to float16
    waveform1 = waveform1.to(device).half()
    waveform2 = waveform2.to(device).half()

    # Extract embeddings
    emb1 = model(waveform1).last_hidden_state.mean(dim=1)
    emb2 = model(waveform2).last_hidden_state.mean(dim=1)

    # Compute cosine similarity
    sim = F.cosine_similarity(emb1, emb2)
    return sim.item()

@torch.no_grad()
def evaluate_speaker_verification(veri_file_path, model):
    """
    Evaluate the speaker verification system using EER, TAR@1%FAR, and Speaker Identification Accuracy.
    """
    y_true, scores = [], []
    model.eval()

    # Read the verification pairs
    with open(veri_file_path, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        parts = line.strip().split()
        label = int(parts[0])
        wav1 = f"/kaggle/input/mastervox1vox2/wav/{parts[1]}"
        wav2 = f"/kaggle/input/mastervox1vox2/wav/{parts[2]}"

        # Get similarity score
        score = verification(model, wav1, wav2)

        y_true.append(label)
        scores.append(score)

        if i % 1000 == 0:
            print(f"Processed {i}/{len(lines)} pairs...")
    # Compute EER
    eer,eer_threshold = compute_eer(y_true, scores)

    # Compute TAR@1% FAR
    tar_at_1_far = compute_tar_at_far(y_true, scores)

    # Compute Speaker Identification Accuracy (using EER threshold)
    accuracy = compute_speaker_identification_accuracy(y_true, scores, eer_threshold)

    return {
        "EER (%)": eer,
        "TAR@1% FAR": tar_at_1_far,
        "Speaker Verification Accuracy (%)": accuracy
    }

# Run evaluation
results_pretrain = evaluate_speaker_verification("/kaggle/input/mastervox1vox2/veri_test2.txt", model_pretrain)
print(results_pretrain)

model loaded on cuda
Processed 0/37611 pairs...
Processed 1000/37611 pairs...
Processed 2000/37611 pairs...
Processed 3000/37611 pairs...
Processed 4000/37611 pairs...
Processed 5000/37611 pairs...
Processed 6000/37611 pairs...
Processed 7000/37611 pairs...
Processed 8000/37611 pairs...
Processed 9000/37611 pairs...
Processed 10000/37611 pairs...
Processed 11000/37611 pairs...
Processed 12000/37611 pairs...
Processed 13000/37611 pairs...
Processed 14000/37611 pairs...
Processed 15000/37611 pairs...
Processed 16000/37611 pairs...
Processed 17000/37611 pairs...
Processed 18000/37611 pairs...
Processed 19000/37611 pairs...
Processed 20000/37611 pairs...
Processed 21000/37611 pairs...
Processed 22000/37611 pairs...
Processed 23000/37611 pairs...
Processed 24000/37611 pairs...
Processed 25000/37611 pairs...
Processed 26000/37611 pairs...
Processed 27000/37611 pairs...
Processed 28000/37611 pairs...
Processed 29000/37611 pairs...
Processed 30000/37611 pairs...
Processed 31000/37611 pairs...


In [ ]:
import torch
import gc

# Delete variables
del model, optimizer, arcface_loss , wavlm
# del arcface_loss
gc.collect()

# Empty CUDA cache
torch.cuda.empty_cache()


## Fine Tuning

In [1]:
import torch
import torchaudio
import torchaudio.transforms as T
import os
import glob
import random
from torch.utils.data import Dataset, DataLoader
from transformers import WavLMModel

# Set Device (CUDA or TPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pydub import AudioSegment
import os
import torch
import numpy as np
import torchaudio
from glob import glob
# Split into training (first 100 speakers) and testing (remaining 18 speakers)
VOXCELEB2_PATH = "/kaggle/input/mastervox1vox2/aac/"
speaker_ids = sorted(os.listdir(VOXCELEB2_PATH))
train_speakers = speaker_ids[:100]
test_speakers = speaker_ids[100:118]
speaker_to_label = {spk: i for i, spk in enumerate(train_speakers)}

def collate_fn(batch):
    """
    Pads variable-length waveforms to the same length in a batch.
    """
    waveforms, labels = zip(*batch) 
    waveforms = [torch.tensor(w) if isinstance(w, np.ndarray) else w for w in waveforms]
    max_length = max([w.shape[-1] for w in waveforms])
    padded_waveforms = torch.stack([
        torch.nn.functional.pad(w, (0, max_length - w.shape[-1])) for w in waveforms
    ])
    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor([w.shape[-1] for w in waveforms])  # Store original lengths

    return padded_waveforms, lengths, labels


class VoxCeleb2Dataset(Dataset):
    def __init__(self, root_dir, speaker_list, speaker_to_label, transform=None):
        self.root_dir = root_dir
        self.speaker_list = speaker_list
        self.speaker_to_label = speaker_to_label
        self.transform = transform
        self.audio_paths = []
        self.labels = []
        i=0
        for speaker in speaker_list:
            speaker_dir = os.path.abspath(os.path.join(root_dir, speaker)) 
            for folder in os.listdir(speaker_dir):
                folder_path = os.path.abspath(os.path.join(speaker_dir, folder)) 
                
                for file in glob(os.path.join(folder_path, "*.m4a")):
                    # print(file)
                    self.audio_paths.append(file)
                    self.labels.append(speaker_to_label[speaker])
        # print(self.audio_paths[0:10])
    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        file_path = self.audio_paths[idx]
        label = self.labels[idx]
    
        if file_path.endswith(".m4a"):
            audio = AudioSegment.from_file(file_path, format="m4a")
            waveform = np.array(audio.get_array_of_samples()).astype(np.float32)
            sr = audio.frame_rate
            waveform /= np.iinfo(audio.array_type).max
    
        else:
            # Directly load .wav files
            waveform, sr = torchaudio.load(file_path)
    
        # Apply transformation (if any)
        if self.transform:
            waveform = self.transform(waveform)
        waveform_tensor = torch.from_numpy(waveform)  # Convert numpy array to PyTorch tensor
        
        max_len = 48000  # Desired length

        # Take the first 48,000 samples (keeping dim=0)
        trimmed_waveform = waveform_tensor[:max_len]  # Shape: [1, 48000]

        
        return trimmed_waveform, label

# Assign unique labels for training and testing speakers separately
train_speaker_to_label = {spk: i for i, spk in enumerate(train_speakers)}
test_speaker_to_label = {spk: i+100 for i, spk in enumerate(test_speakers)}

# Create train and test datasets with separate label mappings
train_dataset = VoxCeleb2Dataset(VOXCELEB2_PATH, train_speakers, train_speaker_to_label)
test_dataset = VoxCeleb2Dataset(VOXCELEB2_PATH, test_speakers, test_speaker_to_label)


train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)


In [3]:
for batch_idx, (waveforms, lengths, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx + 3}:")
    print(f"  Waveforms shape: {waveforms.shape}")  # Should be (batch_size, max_length)
    print(f"  Lengths shape: {lengths.shape}")      # (batch_size,)
    print(f"  Labels shape: {labels.shape}")        # (batch_size,)
    print("-" * 50)
    break  # Print only the first batch


Batch 3:
  Waveforms shape: torch.Size([1, 48000])
  Lengths shape: torch.Size([1])
  Labels shape: torch.Size([1])
--------------------------------------------------


In [29]:
import torch
import torch.nn as nn
# import torch_xla.core.xla_model as xm
from transformers import WavLMModel
from peft import LoraConfig, get_peft_model
import torch.nn.functional as F

# Set Device (TPU or GPU)
# device = xm.xla_device() if "xla" in str(xm.xla_device()) else torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained WavLM Model
model_name = "microsoft/wavlm-base-plus"
wavlm = WavLMModel.from_pretrained(model_name).to(device)

In [22]:
# Apply LoRA to Transformer Layers
config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "intermediate_dense", "output_dense"],  # Apply LoRA to attention layers
)
wavlm = get_peft_model(wavlm, config)

# Modify Classification Head
class SpeakerClassifier(nn.Module):
    def __init__(self, base_model, embedding_dim=768, num_classes=100, verification_mode=False):
        super(SpeakerClassifier, self).__init__()
        self.base_model = base_model
        self.fc = nn.Linear(embedding_dim, num_classes)  # ArcFace will replace this

    def forward(self, input_values, attention_mask=None):
        outputs = self.base_model(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # Shape: (batch, seq_len, 768)
        pooled_output = hidden_states.mean(dim=1)  # Average pooling over time
        logits = self.fc(pooled_output)
        return logits, pooled_output  # Return embeddings for ArcFace

In [6]:
class ArcFaceLoss(nn.Module):
    def __init__(self, in_features, out_features, margin=0.5, scale=30.0):
        super(ArcFaceLoss, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))  # Cosine similarity
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))  # Arc cosine
        target_logits = torch.cos(theta + self.margin)
        logits = self.scale * target_logits
        return F.cross_entropy(logits, labels)


In [7]:
# Initialize Model, Loss, and Optimizer
num_classes = 100
model = SpeakerClassifier(wavlm, num_classes=num_classes).to(device)
arcface_loss = ArcFaceLoss(in_features=768, out_features=num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)


In [34]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())
print("Original")
total_params = count_parameters(model_pretrain)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model_pretrain.parameters() if p.requires_grad):,}")
print("Fine Tune with LoRA")
total_params = count_parameters(model)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Original
Total Parameters: 94,381,936
Trainable Parameters: 94,381,936
Fine Tune with LoRA
Total Parameters: 97,113,044
Trainable Parameters: 2,731,108


In [10]:
from tqdm import tqdm

# Training Loop with Progress Bar
epochs = 1
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
    i=1
    for batch in progress_bar:
        inputs, _, labels = batch  # Ignore lengths
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits, embeddings = model(inputs)
        loss = arcface_loss(embeddings, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=total_loss / (progress_bar.n + 1))
            

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(train_loader)}")


Epoch 1/1: 100%|██████████| 29831/29831 [2:25:37<00:00,  3.41it/s, loss=4.54]  

Epoch 1/1, Loss: 4.53611501594493


In [11]:
torch.save(model.state_dict(), "model_finetuned_on.pth")

In [10]:
import torch
from peft import PeftModel
from transformers import WavLMModel
from peft import LoraConfig, get_peft_model
import torchaudio

In [8]:
@torch.no_grad()  # Disable gradients to reduce memory usage
def verification(model, wav1, wav2):
    """
    Compute similarity between two speaker audio files using WavLM embeddings.
    """
    # Load audio files (Processing on CPU)
    waveform1, sr1 = torchaudio.load(wav1)
    waveform2, sr2 = torchaudio.load(wav2)

    # Resample to 16kHz (Keep on CPU)
    if sr1 != 16000:
        waveform1 = Resample(orig_freq=sr1, new_freq=16000)(waveform1)
    if sr2 != 16000:
        waveform2 = Resample(orig_freq=sr2, new_freq=16000)(waveform2)

    # Move to GPU & Convert to float16
    waveform1 = waveform1.to(device).half()
    waveform2 = waveform2.to(device).half()

    # Extract embeddings
    emb1 = model(waveform1)
    emb2 = model(waveform2)

    # Compute cosine similarity
    sim = F.cosine_similarity(emb1, emb2)
    return sim.item()




In [23]:
@torch.no_grad()
def evaluate_speaker_verification(veri_file_path, model):
    """
    Evaluate the speaker verification system using EER, TAR@1%FAR, and Speaker Identification Accuracy.
    """
    y_true, scores = [], []
    model.eval()

    # Read the verification pairs
    with open(veri_file_path, "r") as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        parts = line.strip().split()
        label = int(parts[0])
        wav1 = f"/kaggle/input/mastervox1vox2/wav/{parts[1]}"
        wav2 = f"/kaggle/input/mastervox1vox2/wav/{parts[2]}"

        # Get similarity score
        score = verification(model, wav1, wav2)

        y_true.append(label)
        scores.append(score)
        if i % 1000 == 0:
            print(f"Processed {i}/{len(lines)} pairs...")
    # Compute EER
    eer,eer_threshold = compute_eer(y_true, scores)

    # Compute TAR@1% FAR
    tar_at_1_far = compute_tar_at_far(y_true, scores)

    # Compute Speaker Identification Accuracy (using EER threshold)
    accuracy = compute_speaker_identification_accuracy(y_true, scores, 1)

    return {
        "EER (%)": eer,
        "TAR@1% FAR": tar_at_1_far,
        "Speaker Verification Accuracy (%)": accuracy
    }

In [21]:
import torch
from peft import PeftModel, get_peft_model
from transformers import WavLMModel
import torch.nn as nn

# Set device (automatically detects GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize the base WavLM model and move to device
model_name = "microsoft/wavlm-base-plus"
wavlm = WavLMModel.from_pretrained(model_name).to(device)

# Apply LoRA configuration
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "intermediate_dense", "output_dense"],
)
wavlm = get_peft_model(wavlm, config).to(device)  # Ensure LoRA is on device

# Define SpeakerClassifier (modified for embeddings)
class SpeakerClassifier(nn.Module):
    def __init__(self, base_model, embedding_dim=768, num_classes=100):
        super().__init__()
        self.base_model = base_model.to(device)  # Already float32
        self.fc = nn.Linear(embedding_dim, num_classes).to(device)
        
    def forward(self, input_values, attention_mask=None):
        # Convert input to float32 if needed
        if input_values.dtype != torch.float32:
            input_values = input_values.float()  # Force float32
        
        outputs = self.base_model(input_values, attention_mask=attention_mask)
        embeddings = outputs[0].mean(dim=1)
        return embeddings.float()  # Ensure output is float32

# Initialize and load pretrained weights
model = SpeakerClassifier(wavlm)
model.load_state_dict(torch.load("/kaggle/input/finetunedvox/pytorch/default/1/model_finetuned_on.pth", map_location=device))  # Load directly to device
model.eval()

# Run evaluation
results_finetuned = evaluate_speaker_verification("/kaggle/input/mastervox1vox2/veri_test2.txt", model)
print(results_finetuned)

Using device: cuda


<ipython-input-21-8c25c9ea29f9>:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/input/finetunedvox/pytorch/default/1/model_finetu

Processed 0/37611 pairs...
Processed 1000/37611 pairs...
Processed 2000/37611 pairs...
Processed 3000/37611 pairs...
Processed 4000/37611 pairs...
Processed 5000/37611 pairs...
Processed 6000/37611 pairs...
Processed 7000/37611 pairs...
Processed 8000/37611 pairs...
Processed 9000/37611 pairs...
Processed 10000/37611 pairs...
Processed 11000/37611 pairs...
Processed 12000/37611 pairs...
Processed 13000/37611 pairs...
Processed 14000/37611 pairs...
Processed 15000/37611 pairs...
Processed 16000/37611 pairs...
Processed 17000/37611 pairs...
Processed 18000/37611 pairs...
Processed 19000/37611 pairs...
Processed 20000/37611 pairs...
Processed 21000/37611 pairs...
Processed 22000/37611 pairs...
Processed 23000/37611 pairs...
Processed 24000/37611 pairs...
Processed 25000/37611 pairs...
Processed 26000/37611 pairs...
Processed 27000/37611 pairs...
Processed 28000/37611 pairs...
Processed 29000/37611 pairs...
Processed 30000/37611 pairs...
Processed 31000/37611 pairs...
Processed 32000/37611